[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_45_Track4_Capstone_Multimodal_Agent.ipynb)

# Lesson 45 — Track 4 Capstone: Multimodal Content Pipeline Agent

**Prerequisites:** L42 (Voice Pipelines), L43 (Image Generation), L44 (Document AI)

---

## What You'll Build

Over the past three lessons you built three separate modality tools:

| Lesson | Tool | Input → Output |
|--------|------|----------------|
| L42 | Voice Pipeline | Audio → Text → Audio (ASR + LLM + TTS) |
| L43 | Image Generation | Text prompt → Image (DALL-E 3 / SDXL) |
| L44 | Document AI | PDF → Structured data / Summary |

This capstone **wires all three together** into a `MultimodalAgent` that can handle real-world tasks no single-modality tool can:

- *"Analyze this invoice and visualize the products"* → PDF extract + image gen + voice summary
- *"Turn this research paper into a brief I can listen to"* → PDF analysis + voice narration + concept illustration
- *"What's in this document? Explain it to me."* → Document AI + Claude reasoning + TTS playback

**Design principle:** Each modality is a _layer_ (VoiceLayer, DocumentLayer, ImageLayer). The `MultimodalAgent` orchestrates across layers based on what inputs arrive and what outputs are requested. A `ModalityRouter` makes routing decisions automatically.

## Architecture

```
┌──────────────────────────────────────────────────────────────────┐
│                     MULTIMODAL AGENT                             │
│                                                                  │
│  ┌─────────────┐   ┌──────────────┐   ┌──────────────────────┐  │
│  │ VOICE LAYER │   │ DOCUMENT AI  │   │   IMAGE GEN LAYER    │  │
│  │             │   │    LAYER     │   │                      │  │
│  │ Whisper ASR │   │ pdfplumber + │   │ DALL-E 3 (API)       │  │
│  │ gTTS / TTS  │   │ Claude PDF   │   │ SDXL-Turbo (local)   │  │
│  │             │   │ Pydantic ext │   │ vision describe      │  │
│  └──────┬──────┘   └──────┬───────┘   └──────────┬───────────┘  │
│         │                 │                       │              │
│         └─────────────────┴───────────────────────┘             │
│                           │                                     │
│                  ┌────────▼────────┐                            │
│                  │  MODALITY       │                            │
│                  │  ORCHESTRATOR   │  ← routes task to layers   │
│                  │  (routes tasks) │    based on inputs/outputs  │
│                  └────────┬────────┘                            │
│                           │                                     │
│                  ┌────────▼────────┐                            │
│                  │  COST TRACKER   │                            │
│                  │  (per modality) │                            │
│                  └─────────────────┘                            │
└──────────────────────────────────────────────────────────────────┘
```

**Key design choices:**
1. Each layer is independently callable — useful standalone AND via the agent
2. The orchestrator inspects the `MultimodalTask` to decide which layers fire
3. Every LLM call goes through a single cost-tracking chokepoint (L22/L31 pattern)
4. Layers gracefully degrade: if DALL-E fails, the image step is skipped, not the whole pipeline

In [ ]:
# ── SETUP ─────────────────────────────────────────────────────────────
# GPU runtime recommended (T4): Runtime → Change runtime type → T4
# This installs everything needed across all 3 modalities.

!pip install anthropic openai pdfplumber fpdf2 gtts pillow matplotlib \
             openai-whisper pydantic tabulate ipython -q

# Optional: Hugging Face for SDXL-Turbo (skip if no GPU or want fast run)
# !pip install diffusers transformers accelerate torch -q

print("✅ All dependencies installed")

In [ ]:
# ── IMPORTS & API KEYS ────────────────────────────────────────────────
import anthropic
import openai
import os
import json
import base64
import time
import io
import re
import math
import tempfile
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, Literal
from enum import Enum

import pdfplumber
from fpdf import FPDF
from gtts import gTTS
from IPython.display import Audio, Image as IPImage, display
from pydantic import BaseModel, Field
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.image as mpimg
from PIL import Image

# ── API keys via Colab Secrets (or env vars for local) ────────────────
try:
    from google.colab import userdata
    ANTHROPIC_KEY = userdata.get('ANTHROPIC_API_KEY')
    OPENAI_KEY    = userdata.get('OPENAI_API_KEY')
except Exception:
    ANTHROPIC_KEY = os.environ.get('ANTHROPIC_API_KEY', 'YOUR_ANTHROPIC_KEY')
    OPENAI_KEY    = os.environ.get('OPENAI_API_KEY',    'YOUR_OPENAI_KEY')

claude  = anthropic.Anthropic(api_key=ANTHROPIC_KEY)
oai     = openai.OpenAI(api_key=OPENAI_KEY)

HAIKU   = 'claude-haiku-4-5-20251001'
SONNET  = 'claude-sonnet-4-6'

# ── Cost table (per 1M tokens / per image / per call) ─────────────────
COST = {
    'haiku_in':    0.80,   # $/MTok
    'haiku_out':   4.00,
    'sonnet_in':   3.00,
    'sonnet_out': 15.00,
    'dalle3_std':  0.040,  # $/image
    'dalle3_hd':   0.080,
    'whisper':     0.006,  # $/minute of audio
    'gtts':        0.000,  # free
}

print("✅ Imports complete")
print(f"   Anthropic key: {'*' * 8 + ANTHROPIC_KEY[-4:] if len(ANTHROPIC_KEY) > 8 else 'NOT SET'}")
print(f"   OpenAI key:    {'*' * 8 + OPENAI_KEY[-4:] if len(OPENAI_KEY) > 8 else 'NOT SET (image gen will be skipped)'}")

## Part 1 — The Three Layers

Each layer is a self-contained class. Before we build the orchestrator, we'll implement and verify each one.
This also gives you clean building blocks you can reuse independently.

### Layer 1: VoiceLayer (ASR + TTS)

Ports the L42 pipeline into a clean class:
- `transcribe(audio_path)` → text  (Whisper or simulated)
- `speak(text)` → audio bytes  (gTTS)
- `make_test_audio(text)` → audio file path  (for Colab testing)

In [ ]:
# ── VOICE LAYER ───────────────────────────────────────────────────────

@dataclass
class VoiceResult:
    text: str
    latency_s: float
    cost_usd: float
    method: Literal['whisper', 'gtts_reverse', 'simulation'] = 'simulation'


class VoiceLayer:
    """ASR + TTS layer. Wraps Whisper (transcription) and gTTS (synthesis)."""

    def __init__(self, whisper_model: str = 'base', use_whisper: bool = True):
        self.whisper_model = whisper_model
        self.use_whisper   = use_whisper
        self._whisper      = None  # lazy-load

    def _load_whisper(self):
        if self._whisper is None:
            import whisper
            print(f"   Loading Whisper ({self.whisper_model})…")
            self._whisper = whisper.load_model(self.whisper_model)
            print("   ✅ Whisper ready")
        return self._whisper

    def make_test_audio(self, text: str, path: str = '/tmp/test_input.mp3') -> str:
        """Generate synthetic 'user speech' for testing in Colab."""
        tts = gTTS(text=text, lang='en', slow=False)
        tts.save(path)
        return path

    def transcribe(self, audio_path: str) -> VoiceResult:
        """Convert audio file → text using Whisper."""
        t0 = time.time()
        if self.use_whisper:
            model = self._load_whisper()
            result = model.transcribe(audio_path, temperature=0.0, fp16=False)
            text = result['text'].strip()
            # Estimate audio duration from file size (rough)
            size_mb = Path(audio_path).stat().st_size / 1e6
            est_minutes = size_mb / 1.0  # ~1 MB/min for MP3
            cost = est_minutes * COST['whisper']
            method = 'whisper'
        else:
            # Simulation: read the file path as a hint
            text = "[Simulated ASR — enable Whisper for real transcription]"
            cost = 0.0
            method = 'simulation'
        return VoiceResult(text=text, latency_s=time.time()-t0, cost_usd=cost, method=method)

    def speak(self, text: str, path: str = '/tmp/tts_output.mp3') -> str:
        """Convert text → speech using gTTS. Returns path to audio file."""
        # Clip to reasonable length for TTS
        if len(text) > 500:
            text = text[:497] + '...'
        tts = gTTS(text=text, lang='en', slow=False)
        tts.save(path)
        return path

    def play(self, audio_path: str):
        """Display audio widget in Colab."""
        display(Audio(audio_path, autoplay=False))


# ── Quick test ────────────────────────────────────────────────────────
voice = VoiceLayer(whisper_model='base', use_whisper=True)

test_text = "Analyze this invoice and tell me the total amount due."
audio_path = voice.make_test_audio(test_text)
print(f"✅ Generated test audio: {audio_path}")

asr_result = voice.transcribe(audio_path)
print(f"\nASR transcription: '{asr_result.text}'")
print(f"Latency: {asr_result.latency_s:.2f}s  |  Cost: ${asr_result.cost_usd:.4f}")

# Play the test audio
voice.play(audio_path)

# 💡 EXPERIMENT: Try make_test_audio with a longer sentence and watch Whisper WER

In [ ]:
# ── DOCUMENT LAYER ────────────────────────────────────────────────────
# Ports the L44 Document AI pipeline into a clean layer class.

class InvoiceLineItem(BaseModel):
    description: str
    quantity: int = Field(default=1, ge=1)
    unit_price: float = Field(ge=0)
    total: float = Field(ge=0)

class InvoiceExtract(BaseModel):
    vendor_name: str
    invoice_number: str
    invoice_date: str
    total_amount: float
    currency: str = 'USD'
    line_items: list[InvoiceLineItem] = []
    payment_terms: Optional[str] = None


@dataclass
class DocResult:
    text: str                  # raw extracted text
    structured: Optional[dict] # pydantic .model_dump() or None
    model_used: str
    input_tokens: int
    output_tokens: int
    cost_usd: float
    latency_s: float
    method: Literal['pdfplumber', 'claude_pdf', 'hybrid'] = 'hybrid'


def _compute_cost(model: str, inp: int, out: int) -> float:
    if 'haiku' in model:
        return (inp * COST['haiku_in'] + out * COST['haiku_out']) / 1_000_000
    return (inp * COST['sonnet_in'] + out * COST['sonnet_out']) / 1_000_000


class DocumentLayer:
    """PDF extraction + Claude-powered analysis and structured extraction."""

    def __init__(self, default_model: str = SONNET):
        self.default_model = default_model

    def create_sample_invoice(self, path: str = '/tmp/sample_invoice.pdf') -> str:
        """Generate a realistic sample invoice PDF for testing."""
        pdf = FPDF()
        pdf.add_page()
        pdf.set_font('Helvetica', 'B', 20)
        pdf.cell(0, 10, 'INVOICE', ln=True, align='C')
        pdf.set_font('Helvetica', '', 11)
        pdf.ln(5)
        pdf.cell(0, 8, 'Acme Software Solutions Inc.', ln=True)
        pdf.cell(0, 8, '123 Innovation Drive, San Francisco CA 94105', ln=True)
        pdf.cell(0, 8, 'invoice@acmesoftware.io', ln=True)
        pdf.ln(5)
        pdf.set_font('Helvetica', 'B', 11)
        pdf.cell(90, 8, 'Invoice Number: INV-2026-0042', ln=False)
        pdf.cell(90, 8, 'Date: June 15, 2026', ln=True)
        pdf.cell(90, 8, 'Due Date: July 15, 2026', ln=False)
        pdf.cell(90, 8, 'Client: TechCorp Ltd.', ln=True)
        pdf.ln(5)
        # Table header
        pdf.set_fill_color(220, 220, 220)
        pdf.set_font('Helvetica', 'B', 10)
        pdf.cell(90, 8, 'Description', border=1, fill=True)
        pdf.cell(25, 8, 'Qty', border=1, fill=True, align='C')
        pdf.cell(37, 8, 'Unit Price', border=1, fill=True, align='C')
        pdf.cell(38, 8, 'Total', border=1, fill=True, align='C')
        pdf.ln()
        # Line items
        pdf.set_font('Helvetica', '', 10)
        items = [
            ('AI Agent Development Platform (monthly)', 1, 2500.00),
            ('Vector Database Storage (100GB)', 3, 150.00),
            ('LLM API Credits — 10M tokens', 2, 400.00),
            ('Dedicated Support Hours', 5, 200.00),
        ]
        subtotal = 0.0
        for desc, qty, price in items:
            total = qty * price
            subtotal += total
            pdf.cell(90, 8, desc, border=1)
            pdf.cell(25, 8, str(qty), border=1, align='C')
            pdf.cell(37, 8, f'${price:,.2f}', border=1, align='R')
            pdf.cell(38, 8, f'${total:,.2f}', border=1, align='R')
            pdf.ln()
        tax = subtotal * 0.085
        grand = subtotal + tax
        pdf.ln(3)
        pdf.set_font('Helvetica', 'B', 10)
        pdf.cell(152, 8, 'Subtotal:', align='R')
        pdf.cell(38, 8, f'${subtotal:,.2f}', border=1, align='R'); pdf.ln()
        pdf.cell(152, 8, 'Tax (8.5%):', align='R')
        pdf.cell(38, 8, f'${tax:,.2f}', border=1, align='R'); pdf.ln()
        pdf.set_font('Helvetica', 'B', 12)
        pdf.cell(152, 10, 'TOTAL DUE:', align='R')
        pdf.cell(38, 10, f'${grand:,.2f}', border=1, align='R'); pdf.ln()
        pdf.ln(8)
        pdf.set_font('Helvetica', '', 9)
        pdf.cell(0, 6, 'Payment terms: Net 30. Wire transfer to: Routing 021000021 / Account 4567890123', ln=True)
        pdf.output(path)
        return path

    def _pdf_to_base64(self, path: str) -> str:
        with open(path, 'rb') as f:
            return base64.standard_b64encode(f.read()).decode('utf-8')

    def extract_text(self, pdf_path: str) -> str:
        """Fast pdfplumber extraction (no LLM, free)."""
        with pdfplumber.open(pdf_path) as pdf:
            return '\n'.join(page.extract_text() or '' for page in pdf.pages)

    def analyze(self, pdf_path: str, question: str) -> DocResult:
        """Send PDF to Claude for free-form question answering."""
        t0 = time.time()
        b64 = self._pdf_to_base64(pdf_path)
        resp = claude.messages.create(
            model=self.default_model,
            max_tokens=1024,
            messages=[{
                'role': 'user',
                'content': [
                    {'type': 'document',
                     'source': {'type': 'base64', 'media_type': 'application/pdf', 'data': b64}},
                    {'type': 'text', 'text': question}
                ]
            }]
        )
        usage = resp.usage
        cost  = _compute_cost(self.default_model, usage.input_tokens, usage.output_tokens)
        return DocResult(
            text=resp.content[0].text,
            structured=None,
            model_used=self.default_model,
            input_tokens=usage.input_tokens,
            output_tokens=usage.output_tokens,
            cost_usd=cost,
            latency_s=time.time()-t0,
            method='claude_pdf'
        )

    def extract_invoice(self, pdf_path: str) -> DocResult:
        """Structured extraction → InvoiceExtract Pydantic model."""
        t0    = time.time()
        b64   = self._pdf_to_base64(pdf_path)
        tool  = {
            'name': 'extract_invoice',
            'description': 'Extract all invoice fields from the document.',
            'input_schema': InvoiceExtract.model_json_schema()
        }
        resp = claude.messages.create(
            model=self.default_model,
            max_tokens=1024,
            tools=[tool],
            tool_choice={'type': 'tool', 'name': 'extract_invoice'},
            messages=[{
                'role': 'user',
                'content': [
                    {'type': 'document',
                     'source': {'type': 'base64', 'media_type': 'application/pdf', 'data': b64}},
                    {'type': 'text',
                     'text': 'Extract every field from this invoice into the tool schema.'}
                ]
            }]
        )
        raw_input = next(b.input for b in resp.content if b.type == 'tool_use')
        parsed    = InvoiceExtract.model_validate(raw_input)
        usage     = resp.usage
        cost      = _compute_cost(self.default_model, usage.input_tokens, usage.output_tokens)
        return DocResult(
            text=f"Invoice #{parsed.invoice_number} — Total: ${parsed.total_amount:,.2f}",
            structured=parsed.model_dump(),
            model_used=self.default_model,
            input_tokens=usage.input_tokens,
            output_tokens=usage.output_tokens,
            cost_usd=cost,
            latency_s=time.time()-t0,
            method='claude_pdf'
        )


# ── Quick test ────────────────────────────────────────────────────────
doc_layer = DocumentLayer(default_model=SONNET)
invoice_path = doc_layer.create_sample_invoice()
print(f"✅ Created sample invoice: {invoice_path}")

print("\n── Extracting invoice text via pdfplumber ──")
raw_text = doc_layer.extract_text(invoice_path)
print(raw_text[:300])

print("\n── Structured extraction via Claude PDF block ──")
inv_result = doc_layer.extract_invoice(invoice_path)
inv_data   = inv_result.structured
print(f"Vendor: {inv_data['vendor_name']}")
print(f"Invoice #: {inv_data['invoice_number']}")
print(f"Total: ${inv_data['total_amount']:,.2f}")
print(f"Line items: {len(inv_data['line_items'])}")
print(f"Cost: ${inv_result.cost_usd:.4f}  |  Latency: {inv_result.latency_s:.1f}s")

# 💡 EXPERIMENT: Call doc_layer.analyze(invoice_path, 'What are the payment details?')

In [ ]:
# ── IMAGE LAYER ───────────────────────────────────────────────────────
# Ports the L43 Image Generation pipeline into a clean layer class.

@dataclass
class ImageResult:
    image_bytes: Optional[bytes]   # PNG/JPEG bytes, None if failed
    revised_prompt: str
    description: str               # Claude's vision description
    cost_usd: float
    latency_s: float
    backend: Literal['dalle3', 'sdxl', 'failed'] = 'dalle3'


class ImageLayer:
    """Image generation layer — DALL-E 3 primary, graceful fail on API key missing."""

    STYLE_SUFFIXES = {
        'photorealistic': 'photorealistic, 8K, high detail, natural lighting',
        'illustration':   'flat illustration, clean vector art, vibrant colors',
        'technical':      'technical diagram, blueprint style, clean lines, labels',
        'minimalist':     'minimalist, white background, simple shapes, monochrome',
        'infographic':    'infographic style, icons, data visualization, professional',
    }

    def __init__(self, quality: Literal['standard', 'hd'] = 'standard'):
        self.quality = quality
        self._has_openai = OPENAI_KEY not in ('', 'YOUR_OPENAI_KEY')
        if not self._has_openai:
            print("⚠️  No OpenAI key — ImageLayer will return placeholder images")

    def _expand_prompt(self, brief: str, style: str) -> str:
        """Use Haiku to expand a short description into a rich DALL-E prompt."""
        suffix = self.STYLE_SUFFIXES.get(style, '')
        resp = claude.messages.create(
            model=HAIKU,
            max_tokens=200,
            messages=[{'role': 'user', 'content':
                f'Expand this into a rich DALL-E image prompt (2 sentences max). '
                f'Style: {style}. Subject: {brief}. '
                f'End with: {suffix}. Output only the prompt text.'}]
        )
        return resp.content[0].text.strip()

    def _describe_image(self, image_bytes: bytes) -> str:
        """Use Claude vision to describe the generated image."""
        b64 = base64.standard_b64encode(image_bytes).decode()
        resp = claude.messages.create(
            model=HAIKU,
            max_tokens=150,
            messages=[{'role': 'user', 'content': [
                {'type': 'image', 'source': {'type': 'base64',
                  'media_type': 'image/png', 'data': b64}},
                {'type': 'text', 'text': 'Describe this image in 2 sentences for a voice assistant.'}
            ]}]
        )
        return resp.content[0].text.strip()

    def _make_placeholder(self, text: str) -> bytes:
        """Generate a simple placeholder image when DALL-E is unavailable."""
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.set_facecolor('#f0f4f8')
        ax.text(0.5, 0.5, f'[Image Placeholder]\n\n{text[:80]}',
                ha='center', va='center', fontsize=11,
                transform=ax.transAxes, wrap=True,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        ax.axis('off')
        buf = io.BytesIO()
        fig.savefig(buf, format='png', bbox_inches='tight', dpi=100)
        plt.close(fig)
        buf.seek(0)
        return buf.read()

    def generate(self, description: str, style: str = 'illustration',
                 size: str = '1024x1024', expand_prompt: bool = True) -> ImageResult:
        """Generate an image from a text description."""
        t0 = time.time()

        # Expand prompt via Haiku
        if expand_prompt:
            prompt = self._expand_prompt(description, style)
        else:
            prompt = description

        if not self._has_openai:
            # Return placeholder without hitting DALL-E
            placeholder = self._make_placeholder(description)
            return ImageResult(
                image_bytes=placeholder,
                revised_prompt=prompt,
                description=f"A placeholder visualization for: {description}",
                cost_usd=0.0,
                latency_s=time.time()-t0,
                backend='failed'
            )

        try:
            resp = oai.images.generate(
                model='dall-e-3',
                prompt=prompt,
                n=1,
                size=size,
                quality=self.quality,
                response_format='b64_json'   # Never use URL — it expires in 60 min
            )
            img_data = resp.data[0]
            img_bytes  = base64.b64decode(img_data.b64_json)
            revised    = img_data.revised_prompt or prompt
            cost       = COST[f'dalle3_{self.quality[:3]}']

            # Describe via Claude vision for voice narration
            description_text = self._describe_image(img_bytes)

            return ImageResult(
                image_bytes=img_bytes,
                revised_prompt=revised,
                description=description_text,
                cost_usd=cost,
                latency_s=time.time()-t0,
                backend='dalle3'
            )
        except Exception as e:
            print(f"   ⚠️ Image generation failed: {e}")
            placeholder = self._make_placeholder(description)
            return ImageResult(
                image_bytes=placeholder,
                revised_prompt=prompt,
                description=f"Image generation failed — showing placeholder for: {description}",
                cost_usd=0.0,
                latency_s=time.time()-t0,
                backend='failed'
            )

    def show(self, result: ImageResult, title: str = ''):
        """Display image in Colab."""
        if result.image_bytes:
            img = Image.open(io.BytesIO(result.image_bytes))
            plt.figure(figsize=(8, 6))
            plt.imshow(img)
            plt.axis('off')
            if title:
                plt.title(title, fontsize=12)
            plt.tight_layout()
            plt.show()


# ── Quick test ────────────────────────────────────────────────────────
image_layer = ImageLayer(quality='standard')

test_prompt = "AI agent platform software product, modern tech company"
print(f"Generating image for: '{test_prompt}'")
img_result = image_layer.generate(test_prompt, style='illustration')

print(f"Backend: {img_result.backend}")
print(f"Revised prompt: {img_result.revised_prompt[:100]}…")
print(f"Vision description: {img_result.description}")
print(f"Cost: ${img_result.cost_usd:.3f}  |  Latency: {img_result.latency_s:.1f}s")

image_layer.show(img_result, title='Generated from invoice line item')

# 💡 EXPERIMENT: Try style='technical' or style='infographic' — different visual results

## Part 2 — The Multimodal Orchestrator

Now we wire the three layers together. The key design is a `MultimodalTask` dataclass that captures what inputs
are available and what outputs are requested. The `ModalityOrchestrator` inspects it and fires the right layers.

**Routing rules:**

| Input has | Output wants | Layers fired |
|-----------|--------------|-------------|
| audio_path | text | Voice (ASR) |
| pdf_path | summary | Document (analyze) |
| pdf_path + want_structured | structured data | Document (extract_invoice) |
| text/pdf content | image | Image (generate) |
| anything | voice_response | Voice (TTS on final answer) |
| pdf_path + want_image + want_voice | full pipeline | All three layers |

This table is the core of the orchestrator — it reads the task and dispatches accordingly.

In [ ]:
# ── MULTIMODAL TASK ───────────────────────────────────────────────────

@dataclass
class MultimodalTask:
    """Describes what inputs are available and what outputs are wanted."""

    # Inputs
    text_query:   Optional[str]  = None   # user's text question/request
    audio_path:   Optional[str]  = None   # path to audio file (mic input)
    pdf_path:     Optional[str]  = None   # path to PDF document
    image_path:   Optional[str]  = None   # path to an existing image

    # Output flags
    want_structured: bool = False  # extract structured data (e.g. InvoiceExtract)
    want_image:      bool = False  # generate an image
    want_voice:      bool = True   # speak the final answer
    image_style:     str  = 'illustration'

    # Routing hint (optional override)
    image_subject: Optional[str] = None   # explicit subject for image gen


@dataclass
class MultimodalResult:
    """Everything the agent produced for one task."""
    transcript:       Optional[str]        = None   # ASR output
    doc_analysis:     Optional[str]        = None   # Claude answer about doc
    structured_data:  Optional[dict]       = None   # pydantic extract
    image_result:     Optional[ImageResult] = None
    audio_path:       Optional[str]        = None   # TTS output path
    final_answer:     str                  = ''
    total_cost_usd:   float                = 0.0
    total_latency_s:  float                = 0.0
    steps_taken:      list[str]            = field(default_factory=list)


print("✅ Task + Result dataclasses defined")

In [ ]:
# ── MULTIMODAL ORCHESTRATOR ───────────────────────────────────────────

class MultimodalAgent:
    """
    Orchestrates VoiceLayer + DocumentLayer + ImageLayer
    to handle multimodal tasks end-to-end.
    """

    def __init__(
        self,
        voice_layer:    Optional[VoiceLayer]    = None,
        doc_layer:      Optional[DocumentLayer] = None,
        image_layer:    Optional[ImageLayer]    = None,
        max_cost_usd:   float = 1.00,
        verbose:        bool  = True
    ):
        self.voice   = voice_layer   or VoiceLayer()
        self.docs    = doc_layer     or DocumentLayer()
        self.images  = image_layer   or ImageLayer()
        self.max_cost = max_cost_usd
        self.verbose  = verbose

    def _log(self, msg: str):
        if self.verbose:
            print(f"  [{time.strftime('%H:%M:%S')}] {msg}")

    def _budget_check(self, spent: float):
        if spent >= self.max_cost:
            raise RuntimeError(
                f"Budget cap hit: ${spent:.4f} >= ${self.max_cost:.2f}. "
                f"Increase max_cost_usd to continue."
            )

    def run(self, task: MultimodalTask) -> MultimodalResult:
        """Execute the multimodal pipeline for the given task."""
        t0     = time.time()
        result = MultimodalResult()
        cost   = 0.0

        # ── STEP 1: Voice input (ASR) ──────────────────────────────────
        if task.audio_path:
            self._log("🎤 ASR: Transcribing audio…")
            asr = self.voice.transcribe(task.audio_path)
            result.transcript = asr.text
            cost += asr.cost_usd
            result.steps_taken.append('asr')
            # Merge ASR transcript into text_query
            if not task.text_query:
                task.text_query = asr.text
            self._log(f"   Transcript: '{asr.text[:80]}'")
            self._budget_check(cost)

        # ── STEP 2: Document analysis ───────────────────────────────────
        if task.pdf_path:
            question = task.text_query or "Summarize this document concisely."

            if task.want_structured:
                self._log("📄 DOC: Structured extraction (invoice)…")
                doc_r = self.docs.extract_invoice(task.pdf_path)
                result.structured_data = doc_r.structured
            else:
                self._log(f"📄 DOC: Analyzing with question: '{question[:60]}'…")
                doc_r = self.docs.analyze(task.pdf_path, question)

            result.doc_analysis = doc_r.text
            cost += doc_r.cost_usd
            result.steps_taken.append('doc_analysis')
            self._log(f"   Answer: '{doc_r.text[:100]}'…")
            self._budget_check(cost)

        # ── STEP 3: Determine image subject ────────────────────────────
        if task.want_image:
            # Derive subject: explicit > extracted from structured > text query
            if task.image_subject:
                subject = task.image_subject
            elif result.structured_data and result.structured_data.get('line_items'):
                items   = result.structured_data['line_items']
                subject = items[0]['description']  # most prominent line item
            elif result.doc_analysis:
                subject = result.doc_analysis[:100]
            else:
                subject = task.text_query or 'abstract visualization'

            self._log(f"🎨 IMAGE: Generating image for: '{subject[:60]}'…")
            img_r = self.images.generate(subject, style=task.image_style)
            result.image_result = img_r
            cost += img_r.cost_usd
            result.steps_taken.append('image_gen')
            self._log(f"   Backend: {img_r.backend}  |  ${img_r.cost_usd:.3f}")
            self._budget_check(cost)

        # ── STEP 4: Synthesize final answer ────────────────────────────
        parts = []
        if result.transcript:
            parts.append(f"You asked: {result.transcript}")
        if result.structured_data:
            sd = result.structured_data
            parts.append(
                f"Invoice from {sd.get('vendor_name','?')} — "
                f"#{sd.get('invoice_number','?')} — "
                f"Total due: ${sd.get('total_amount',0):,.2f} {sd.get('currency','USD')}. "
                f"{len(sd.get('line_items',[]))} line items."
            )
        elif result.doc_analysis:
            parts.append(result.doc_analysis)
        if result.image_result:
            parts.append(f"I've generated an image: {result.image_result.description}")

        result.final_answer = ' '.join(parts) if parts else 'Task complete.'

        # ── STEP 5: TTS output ─────────────────────────────────────────
        if task.want_voice:
            self._log("🔊 TTS: Converting answer to speech…")
            audio = self.voice.speak(result.final_answer)
            result.audio_path = audio
            result.steps_taken.append('tts')

        result.total_cost_usd  = cost
        result.total_latency_s = time.time() - t0
        self._log(f"✅ Done in {result.total_latency_s:.1f}s | Total cost: ${cost:.4f}")
        return result


# ── Instantiate the agent ─────────────────────────────────────────────
agent = MultimodalAgent(
    voice_layer  = voice,
    doc_layer    = doc_layer,
    image_layer  = image_layer,
    max_cost_usd = 2.00,
    verbose      = True
)

print("✅ MultimodalAgent ready")
print(f"   Layers: Voice({'✅' if voice else '❌'}), "
      f"Docs({'✅' if doc_layer else '❌'}), "
      f"Images({'✅' if image_layer else '❌'})")

## Use Case 1 — Invoice Analysis + Visualization

**Scenario:** You receive an invoice PDF and want to:
1. Extract all fields into structured data
2. Generate a product visualization for the main line item
3. Get a voice summary you can listen to

**Layers fired:** DocumentLayer (structured) → ImageLayer → VoiceLayer (TTS)

This is the most common enterprise use case — automated invoice processing with visual confirmation.

In [ ]:
# ── USE CASE 1: Invoice → Structured + Image + Voice ─────────────────
print("=" * 60)
print("USE CASE 1: Invoice Analysis Pipeline")
print("=" * 60)

task1 = MultimodalTask(
    pdf_path         = invoice_path,
    text_query       = "What are the main products and the total amount due?",
    want_structured  = True,    # ← full InvoiceExtract
    want_image       = True,    # ← generate product visualization
    want_voice       = True,    # ← speak the summary
    image_style      = 'technical',
)

result1 = agent.run(task1)

print("\n── Structured Data ──")
sd = result1.structured_data
if sd:
    print(f"  Vendor:   {sd['vendor_name']}")
    print(f"  Invoice:  {sd['invoice_number']}  ({sd['invoice_date']})")
    print(f"  Total:    ${sd['total_amount']:,.2f}")
    print(f"  Terms:    {sd.get('payment_terms','N/A')}")
    print("  Line Items:")
    for li in sd.get('line_items', []):
        print(f"    • {li['description'][:55]:<55}  qty={li['quantity']}  ${li['total']:,.2f}")

print("\n── Final Answer ──")
print(f"  {result1.final_answer[:300]}")

print("\n── Image Generated ──")
if result1.image_result:
    image_layer.show(result1.image_result, title='Product Visualization (invoice line item 1)')

print("\n── Voice Summary (click play) ──")
if result1.audio_path:
    voice.play(result1.audio_path)

print(f"\n── Cost Breakdown ──")
print(f"  Steps: {' → '.join(result1.steps_taken)}")
print(f"  Total cost:    ${result1.total_cost_usd:.4f}")
print(f"  Total latency: {result1.total_latency_s:.1f}s")

## Use Case 2 — Voice-Driven Document Q&A

**Scenario:** User speaks a question → Whisper transcribes it → Claude answers from the PDF → gTTS speaks the answer.

This is the classic **voice assistant for documents** pattern. Think: "Ask your PDF questions with your voice."

**Layers fired:** VoiceLayer (ASR) → DocumentLayer (analyze) → VoiceLayer (TTS)

**Key lesson:** The ASR output feeds naturally into the `text_query` field, making the routing clean — the orchestrator doesn't need to know it came from voice.

In [ ]:
# ── USE CASE 2: Voice Q&A over a Document ────────────────────────────
print("=" * 60)
print("USE CASE 2: Voice-Driven Document Q&A")
print("=" * 60)

# Simulate user speaking a question
spoken_question = "What are the payment terms and the bank routing details?"
audio_question  = voice.make_test_audio(spoken_question, path='/tmp/question.mp3')
print(f"\nSimulated voice input: '{spoken_question}'")
print("[User audio]")
voice.play(audio_question)

task2 = MultimodalTask(
    audio_path = audio_question,  # ← voice input
    pdf_path   = invoice_path,    # ← document to query
    want_image = False,           # no image this time
    want_voice = True,            # speak the answer back
)

result2 = agent.run(task2)

print(f"\n── ASR Transcript ──")
print(f"  '{result2.transcript}'")

print(f"\n── Claude's Answer ──")
print(f"  {result2.doc_analysis}")

print(f"\n── Voice Response (click play) ──")
if result2.audio_path:
    voice.play(result2.audio_path)

print(f"\n── Cost ──")
print(f"  Steps: {' → '.join(result2.steps_taken)}")
print(f"  ${result2.total_cost_usd:.4f}  |  {result2.total_latency_s:.1f}s")

# 💡 EXPERIMENT: Change the spoken_question and re-run to ask different things about the invoice

In [ ]:
# ── USE CASE 3: Text Query → Image + Voice (no document) ─────────────
# Demonstrates that the agent works with just a text prompt too.
print("=" * 60)
print("USE CASE 3: Text → Image + Voice Narration")
print("=" * 60)

task3 = MultimodalTask(
    text_query    = "Explain how a vector database stores embeddings",
    want_image    = True,
    want_voice    = True,
    image_style   = 'technical',
    image_subject = "vector database architecture showing embedding vectors in high-dimensional space"
)

result3 = agent.run(task3)

print("\n── Generated Image ──")
if result3.image_result:
    image_layer.show(result3.image_result, title='Vector Database Architecture')
    print(f"  Vision description: {result3.image_result.description}")

print("\n── Voice Narration (click play) ──")
if result3.audio_path:
    voice.play(result3.audio_path)

print(f"\n  Steps: {' → '.join(result3.steps_taken)}")
print(f"  Cost: ${result3.total_cost_usd:.4f}  |  {result3.total_latency_s:.1f}s")

# 💡 EXPERIMENT: Change image_style to 'infographic' and compare the output

## Part 3 — Cost Model Across Modalities

Each modality has a different cost profile. Understanding this matters for building agents that don't surprise you with a $50 bill:

| Modality | Tool | Cost per call | Notes |
|----------|------|-------------|-------|
| ASR | Whisper (local) | Free | Only cost is GPU time |
| ASR | OpenAI Whisper API | $0.006/min | Cheapest hosted option |
| Doc analysis | Claude Haiku (PDF) | ~$0.002/page | Haiku for simple Q&A |
| Doc analysis | Claude Sonnet (PDF) | ~$0.015/page | Sonnet for extraction |
| Image gen | DALL-E 3 Standard | $0.040/image | 1024×1024 |
| Image gen | DALL-E 3 HD | $0.080/image | Better for detail |
| Image gen | SDXL local (T4) | ~$0.003/image | Amortized GPU cost |
| TTS | gTTS | Free | Google TTS, rate-limited |
| TTS | ElevenLabs | $0.0003/char | Best quality |

**The expensive part is image generation** — a pipeline that generates 1 image per document page at scale costs ~$40 per 1,000 pages. Always cap it.

In [ ]:
# ── COST TRACKING & REPORTING ─────────────────────────────────────────

@dataclass
class ModalityCostTracker:
    """Track costs per modality across multiple agent runs."""
    _history: list[MultimodalResult] = field(default_factory=list)

    def record(self, result: MultimodalResult):
        self._history.append(result)

    def report(self):
        if not self._history:
            print("No runs recorded.")
            return

        # Compute per-modality cost estimates
        modality_costs = {'asr': 0.0, 'doc_analysis': 0.0,
                          'image_gen': 0.0, 'tts': 0.0, 'other': 0.0}

        for r in self._history:
            if r.image_result:
                modality_costs['image_gen'] += r.image_result.cost_usd
                remainder = r.total_cost_usd - r.image_result.cost_usd
            else:
                remainder = r.total_cost_usd

            if 'asr' in r.steps_taken:
                modality_costs['asr'] += 0.001  # rough Whisper estimate
                remainder -= 0.001
            if 'doc_analysis' in r.steps_taken:
                # Most of the remainder is doc analysis
                modality_costs['doc_analysis'] += max(0, remainder)
                remainder = 0
            modality_costs['other'] += max(0, remainder)

        total = sum(modality_costs.values())
        n     = len(self._history)

        print(f"\n{'='*50}")
        print(f" COST REPORT  ({n} runs)")
        print(f"{'='*50}")
        print(f"  {'Modality':<20} {'Total':>10}  {'Avg/run':>10}  {'Share':>8}")
        print(f"  {'-'*48}")
        for mod, cost in sorted(modality_costs.items(), key=lambda x: -x[1]):
            if cost > 0:
                share = cost / total * 100 if total > 0 else 0
                print(f"  {mod:<20} ${cost:>9.4f}  ${cost/n:>9.4f}  {share:>7.1f}%")
        print(f"  {'-'*48}")
        print(f"  {'TOTAL':<20} ${total:>9.4f}  ${total/n:>9.4f}  {'100.0':>7}%")

        # Bar chart
        fig, ax = plt.subplots(figsize=(8, 3))
        mods  = [m for m, c in modality_costs.items() if c > 0]
        costs = [modality_costs[m] for m in mods]
        colors = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2', '#59a14f']
        bars = ax.barh(mods, costs, color=colors[:len(mods)])
        ax.set_xlabel('Cost (USD)')
        ax.set_title(f'Multimodal Agent Cost Breakdown ({n} runs)')
        for bar, cost in zip(bars, costs):
            ax.text(bar.get_width() + 0.0001, bar.get_y() + bar.get_height()/2,
                    f'${cost:.4f}', va='center', fontsize=9)
        plt.tight_layout()
        plt.show()


# Track the three runs above
tracker = ModalityCostTracker()
tracker.record(result1)
tracker.record(result2)
tracker.record(result3)
tracker.report()

# Key insight:
print("\n💡 KEY INSIGHT: Image generation is the dominant cost in multimodal pipelines.")
print("   Only generate images when explicitly requested — not on every run.")

## Part 4 — Production Patterns

Three patterns every production multimodal agent needs:

### Pattern 1: Graceful degradation
If image generation fails (API error, content policy), continue without it. The document analysis + voice response are still valuable.

### Pattern 2: Lazy layer loading
Don't load Whisper (300 MB model) until audio input actually arrives. Same for SDXL.

### Pattern 3: Budget-aware routing
If the remaining budget is <$0.05, skip image generation and TTS. Return text only.

In [ ]:
# ── SAFE MULTIMODAL AGENT ─────────────────────────────────────────────
# Adds: graceful degradation, budget tiers, full error handling

class SafeMultimodalAgent(MultimodalAgent):
    """
    Production-hardened multimodal agent.
    - Graceful degradation: image/voice steps are optional, never block doc analysis
    - Budget tiers: skips expensive steps when budget is tight
    - Per-step error catching with detailed logging
    """

    IMAGE_COST_THRESHOLD = 0.05   # don't generate image if remaining budget < this

    def run(self, task: MultimodalTask) -> MultimodalResult:
        t0     = time.time()
        result = MultimodalResult()
        cost   = 0.0
        budget = self.max_cost

        # STEP 1: ASR (optional, fail gracefully)
        if task.audio_path:
            try:
                self._log("🎤 ASR…")
                asr = self.voice.transcribe(task.audio_path)
                result.transcript = asr.text
                cost += asr.cost_usd
                if not task.text_query:
                    task.text_query = asr.text
                result.steps_taken.append('asr')
            except Exception as e:
                self._log(f"⚠️ ASR failed: {e} — using text_query if available")

        # STEP 2: Document analysis (core — fail loudly)
        if task.pdf_path:
            question = task.text_query or "Summarize this document."
            try:
                if task.want_structured:
                    self._log("📄 DOC: Structured extraction…")
                    doc_r = self.docs.extract_invoice(task.pdf_path)
                    result.structured_data = doc_r.structured
                else:
                    self._log(f"📄 DOC: Free-form analysis…")
                    doc_r = self.docs.analyze(task.pdf_path, question)
                result.doc_analysis = doc_r.text
                cost += doc_r.cost_usd
                result.steps_taken.append('doc_analysis')
            except Exception as e:
                self._log(f"❌ Doc analysis failed: {e}")
                result.final_answer = f"Document analysis failed: {e}"
                result.total_cost_usd  = cost
                result.total_latency_s = time.time() - t0
                return result

        # STEP 3: Image generation (optional — budget-gated)
        if task.want_image:
            remaining = budget - cost
            if remaining < self.IMAGE_COST_THRESHOLD:
                self._log(f"⚠️ Skipping image: budget too low (${remaining:.3f} remaining)")
            else:
                try:
                    subject = task.image_subject or (
                        result.doc_analysis[:80] if result.doc_analysis else task.text_query
                    ) or 'abstract visualization'
                    self._log(f"🎨 IMAGE: '{subject[:50]}'…")
                    img_r = self.images.generate(subject, style=task.image_style)
                    result.image_result = img_r
                    cost += img_r.cost_usd
                    result.steps_taken.append('image_gen')
                except Exception as e:
                    self._log(f"⚠️ Image gen failed (continuing without): {e}")

        # STEP 4: Build final answer
        parts = []
        if result.structured_data:
            sd = result.structured_data
            parts.append(
                f"Invoice #{sd.get('invoice_number','?')} from {sd.get('vendor_name','?')}. "
                f"Total due: ${sd.get('total_amount',0):,.2f}."
            )
        elif result.doc_analysis:
            parts.append(result.doc_analysis)
        if result.image_result and result.image_result.backend != 'failed':
            parts.append(f"I also generated a visual: {result.image_result.description}")
        result.final_answer = ' '.join(parts) if parts else 'Task complete — no output generated.'

        # STEP 5: TTS (optional — fail gracefully)
        if task.want_voice:
            try:
                self._log("🔊 TTS…")
                audio = self.voice.speak(result.final_answer)
                result.audio_path = audio
                result.steps_taken.append('tts')
            except Exception as e:
                self._log(f"⚠️ TTS failed (text answer still available): {e}")

        result.total_cost_usd  = cost
        result.total_latency_s = time.time() - t0
        self._log(f"✅ Done in {result.total_latency_s:.1f}s | ${cost:.4f}")
        return result


# ── Test the safe agent with a very tight budget ───────────────────────
safe_agent = SafeMultimodalAgent(
    voice_layer  = voice,
    doc_layer    = doc_layer,
    image_layer  = image_layer,
    max_cost_usd = 0.02,   # very tight — will skip image
    verbose      = True
)

print("\n── Test: Low-budget run (should skip image gen) ──")
safe_result = safe_agent.run(MultimodalTask(
    pdf_path        = invoice_path,
    text_query      = "What is the total amount due?",
    want_structured = False,
    want_image      = True,   # requested but budget prevents it
    want_voice      = True,
))
print(f"\n  Steps taken: {safe_result.steps_taken}")
print(f"  Final answer: {safe_result.final_answer[:150]}")
if safe_result.audio_path:
    voice.play(safe_result.audio_path)

## 10 Multimodal Agent Pitfalls

| # | Pitfall | Why it bites | Fix |
|---|---------|-------------|-----|
| 1 | **Generating images on every turn** | $0.04/image × 100 turns = $4 per conversation | Gate image gen behind explicit user request or `want_image=True` |
| 2 | **Storing DALL-E URLs** | URLs expire after 60 minutes | Always use `response_format='b64_json'` and store bytes |
| 3 | **No budget cap** | Parallel multimodal pipelines can 10× costs | Always set `max_cost_usd` and check after each step |
| 4 | **Blocking on slow TTS** | gTTS can take 2-3s; makes latency feel bad | Move TTS to a background task; stream text first |
| 5 | **Base64 PDF in conversation history** | A 5-page PDF = ~300 KB base64 = millions of tokens if kept in history | Never store PDF base64 in message history; re-encode on each call |
| 6 | **Assuming Whisper is fast** | `large-v3` on T4 Colab is 30-50× real-time | Use `base` model for latency, `large-v3` only for accuracy requirements |
| 7 | **No graceful degradation** | One DALL-E 429 error kills the whole pipeline | Wrap each layer in try/except; return partial results |
| 8 | **Routing everything through the orchestrator for text** | Adds latency overhead for simple text queries | Fast-path: if no pdf_path and no audio_path, skip to LLM directly |
| 9 | **Voice output without SSML tuning** | gTTS sounds robotic on long technical answers | Clip to 500 chars; break at sentence boundaries; use ElevenLabs for quality |
| 10 | **No modality-specific cost tracking** | You see total cost but can't tell if images or LLM is responsible | Track cost per modality with `ModalityCostTracker` so you can optimize the right thing |

In [ ]:
# ── PITFALL DEMO: Base64 PDF in history ──────────────────────────────
# Pitfall #5 is the sneakiest. Let's measure the cost difference.

with open(invoice_path, 'rb') as f:
    pdf_bytes = f.read()

b64_pdf = base64.standard_b64encode(pdf_bytes).decode('utf-8')
b64_chars = len(b64_pdf)
# Anthropic counts ~4 chars per token roughly for base64
approx_tokens = b64_chars // 4

print("PITFALL #5: What happens if you store PDF base64 in message history?")
print(f"")
print(f"  PDF file size:        {len(pdf_bytes):,} bytes")
print(f"  Base64 encoding:      {b64_chars:,} chars")
print(f"  Approx token cost:    ~{approx_tokens:,} tokens")
print(f"  Cost if kept in history for 10 turns (Sonnet):")
cost_in_history = approx_tokens * 10 * COST['sonnet_in'] / 1_000_000
cost_per_call   = approx_tokens * 1  * COST['sonnet_in'] / 1_000_000
print(f"    Re-encode each call:  ${cost_per_call:.4f} × N calls (linear)")
print(f"    Keep in history:      ${cost_in_history:.4f} for 10 turns alone")
print(f"")
print(f"  ✅ CORRECT: Re-encode the PDF bytes fresh on each API call.")
print(f"  ❌ WRONG:   Append the base64 string to messages[] — costs explode.")

## Homework — 5 Extension Tasks

**1. Research Paper → Slide Brief**
Upload a real research paper PDF. Use `doc_layer.analyze()` to extract the 3 key findings, generate one illustration per finding (3 DALL-E calls), and produce a voice narration for each. Budget cap: $0.20 total.

**2. Multi-page Invoice Batch Processing**
Modify `DocumentLayer.extract_invoice()` to handle a folder of PDFs. Use `asyncio.gather` (L35 pattern) to extract all invoices in parallel. Add a `total_across_invoices` aggregation function.

**3. Modality Router with Intent Detection**
Add an intent detection step: before routing, use Haiku to classify the user's voice/text input into one of `[invoice_extraction, doc_qa, image_request, general_chat]`. Use the intent to set the `MultimodalTask` flags automatically — no manual flag setting by the user.

**4. Add SDXL as Image Layer Backend**
Modify `ImageLayer.generate()` to check if a GPU is available via `torch.cuda.is_available()`. If yes, use SDXL-Turbo locally (saving $0.04/image). If no, fall back to DALL-E 3. Use the L43 pattern for SDXL.

**5. FastAPI Endpoint**
Wrap the `SafeMultimodalAgent` in a FastAPI endpoint at `POST /analyze` that:
- Accepts a file upload (PDF) + optional `question` parameter
- Returns JSON with `structured_data`, `summary`, and `image_description`
- Streams the TTS audio back as `audio/mp3`
Test it with `httpx` in Colab using the L19/L23 pattern.

In [ ]:
# ── TRACK 4 COMPLETE — SUMMARY ────────────────────────────────────────
print("\n" + "="*60)
print("  🎉 TRACK 4 COMPLETE: Voice + Multimodal Agents")
print("="*60)

summary = [
    ("L42", "Voice Pipelines",
     "Whisper ASR + gTTS + VoiceAgent class + streaming latency budget"),
    ("L43", "Image Generation",
     "DALL-E 3 b64 + SDXL-Turbo + prompt engineering + vision-describe loop"),
    ("L44", "Document AI",
     "pdfplumber + Claude PDF block + Pydantic extraction + map-reduce over long docs"),
    ("L45", "Track 4 Capstone",
     "MultimodalAgent: VoiceLayer + DocumentLayer + ImageLayer + cost tracking"),
]

for lesson, title, content in summary:
    print(f"\n  {lesson}: {title}")
    print(f"  {'─'*55}")
    print(f"  {content}")

# Print cost summary from tracker
print("\n" + "="*60)
print("  Cost Summary for This Session")
print("="*60)
total_cost = result1.total_cost_usd + result2.total_cost_usd + result3.total_cost_usd
print(f"  Use Case 1 (invoice + image + voice):  ${result1.total_cost_usd:.4f}")
print(f"  Use Case 2 (voice Q&A on document):    ${result2.total_cost_usd:.4f}")
print(f"  Use Case 3 (text → image + voice):     ${result3.total_cost_usd:.4f}")
print(f"  {'─'*45}")
print(f"  TOTAL:                                 ${total_cost:.4f}")

## What's Next — Track 5: Agent-Ops & Infrastructure

You've now completed four tracks:

| Track | Focus | Status |
|-------|-------|--------|
| Track 1 | Reliability & Safety | ✅ Complete (L24–L31) |
| Track 2 | Multi-Agent Coordination | ✅ Complete (L32–L36) |
| Track 3 | Self-hosted LLMs & Fine-tuning | ✅ Complete (L37–L41) |
| Track 4 | Voice + Multimodal Agents | ✅ Complete (L42–L45) |
| **Track 5** | **Agent-Ops & Infrastructure** | **← Next** |

**Track 5 will cover the production deployment and ops layer:**

- **L46:** Durable execution with Temporal/Inngest — long-running agent tasks that survive crashes, restarts, and timeouts. The answer to "what happens when your 2-hour agent job dies at step 47?"
- **L47:** GPU autoscaling — serving your fine-tuned models cost-effectively; scale-to-zero on Runpod/Modal/Replicate; spot instance strategies
- **L48:** OpenTelemetry for LLM agents — distributed tracing across A2A agent chains, latency waterfall for multimodal pipelines, cost attribution per trace
- **L49:** Agent evaluation at scale — CI pipelines that run hundreds of eval cases, regression dashboards, automatic quality gates
- **L50:** Track 5 Capstone — fully instrumented, durable, auto-scaling agent system

**The goal of Track 5:** Turn everything you've built across Tracks 1–4 into something you can actually ship to production and operate at scale. This is where you become a _real_ AI engineer, not just someone who can build agents that work on a laptop.

---

*Next lesson: L46 — Durable Execution with Temporal/Inngest*